# 기프트코 FAQ RAG v0 Starter Notebook

목표는 처음부터 복잡한 RAG 서비스를 만드는 것이 아니라, 아래 순서로 **검증 가능한 v0**를 만드는 것입니다.

1. FAQ 엑셀 로딩
2. 컬럼 구조 확인
3. FAQ 데이터를 RAG용 문서 형태로 정리
4. 키워드/문자 n-gram 기반 검색 Top-K 확인
5. 검색 결과를 근거로 답변 초안 생성
6. 실패 케이스를 기록할 평가 파일 생성

> 이 v0는 “답변 생성”보다 먼저 “검색이 제대로 되는지”를 확인하기 위한 노트북입니다.


## 0. 사용 방법

1. 이 노트북과 같은 폴더 또는 `data/` 폴더에 FAQ 엑셀 파일을 넣습니다.
2. 아래 `FAQ_PATH` 값을 실제 파일명으로 수정합니다.
3. 위에서부터 순서대로 실행합니다.
4. 검색 결과 Top-K가 잘 나오는지 먼저 확인합니다.
5. 그 다음 LLM 답변 생성, Chroma/Qdrant, FastAPI로 확장합니다.


In [ ]:
# 필요 패키지 설치
# 처음 한 번만 실행하면 됩니다.
# 주피터 환경에 따라 앞에 !를 붙여 실행하세요.

# !pip install pandas openpyxl scikit-learn python-dotenv openai


In [ ]:
from pathlib import Path
import re
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)


## 1. 파일 경로 설정

`FAQ_PATH`만 실제 파일명에 맞게 수정하면 됩니다.

예시:

```python
FAQ_PATH = Path("data/기프트코_FAQ.xlsx")
```


In [ ]:
# =========================
# 1. 파일 경로
# =========================

FAQ_PATH = Path("../data/기프트코_FAQ.xlsx")  # 여기를 실제 FAQ 엑셀 파일명으로 수정
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NORMALIZED_FAQ_PATH = OUTPUT_DIR / "giftco_faq_normalized_v0.xlsx"
SEARCH_TEST_RESULT_PATH = OUTPUT_DIR / "giftco_faq_search_test_result_v0.xlsx"
EVAL_TEMPLATE_PATH = OUTPUT_DIR / "giftco_faq_eval_template_v0.xlsx"

print("FAQ_PATH:", FAQ_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

## 2. FAQ 엑셀 로딩 및 컬럼 확인

FAQ 파일의 실제 컬럼명이 다를 수 있으므로, 먼저 컬럼명을 확인합니다.


In [ ]:
# =========================
# 2. FAQ 엑셀 로딩
# =========================

if not FAQ_PATH.exists():
    raise FileNotFoundError(
        f"FAQ 파일을 찾을 수 없습니다: {FAQ_PATH}\n"
        "FAQ_PATH를 실제 파일 경로로 수정하세요."
    )

df_raw = pd.read_excel(FAQ_PATH, dtype=str)
df_raw = df_raw.fillna("")

print("행 수:", len(df_raw))
print("컬럼 목록:")
for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i:02d}. {col}")

display(df_raw.head(3))


## 3. 컬럼 자동 매핑

아래 후보 목록을 기준으로 질문/답변/카테고리/키워드/출처/수정일 컬럼을 자동으로 찾습니다.

자동 매핑이 틀리면 직접 컬럼명을 지정하면 됩니다.


In [ ]:
# =========================
# 3. 컬럼 자동 매핑
# =========================

def normalize_col_name(col: str) -> str:
    return re.sub(r"\s+", "", str(col)).lower()

def find_column(df: pd.DataFrame, candidates: list[str], required: bool = False):
    normalized_map = {normalize_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_col_name(cand)
        if key in normalized_map:
            return normalized_map[key]
    if required:
        raise KeyError(
            f"필수 컬럼을 찾지 못했습니다. 후보: {candidates}\n"
            f"현재 컬럼: {list(df.columns)}"
        )
    return None

QUESTION_COL = find_column(df_raw, ["질문", "FAQ질문", "문의", "문의내용", "Q", "Question"], required=True)
ANSWER_COL = find_column(df_raw, ["답변", "FAQ답변", "응답", "답변내용", "A", "Answer"], required=True)
CATEGORY_COL = find_column(df_raw, ["카테고리", "분류", "구분", "유형", "Category"], required=False)
KEYWORD_COL = find_column(df_raw, ["키워드", "관련키워드", "태그", "검색어", "Keyword"], required=False)
SOURCE_COL = find_column(df_raw, ["출처", "원본", "파일명", "Source"], required=False)
UPDATED_AT_COL = find_column(df_raw, ["수정일", "업데이트일", "작성일", "등록일", "UpdatedAt"], required=False)

column_map = {
    "QUESTION_COL": QUESTION_COL,
    "ANSWER_COL": ANSWER_COL,
    "CATEGORY_COL": CATEGORY_COL,
    # "KEYWORD_COL": KEYWORD_COL,
    # "SOURCE_COL": SOURCE_COL,
    # "UPDATED_AT_COL": UPDATED_AT_COL,
}

column_map


### 컬럼 매핑이 틀린 경우

위 결과가 틀리면 아래 셀에서 직접 수정하세요.

예시:

```python
QUESTION_COL = "자주 묻는 질문"
ANSWER_COL = "답변 내용"
CATEGORY_COL = "분류"
```


In [ ]:
# 자동 매핑이 틀린 경우 여기에서 직접 수정하세요.
# 예:
# QUESTION_COL = "자주 묻는 질문"
# ANSWER_COL = "답변 내용"
# CATEGORY_COL = "분류"

print("질문 컬럼:", QUESTION_COL)
print("답변 컬럼:", ANSWER_COL)
print("카테고리 컬럼:", CATEGORY_COL)
# print("키워드 컬럼:", KEYWORD_COL)
# print("출처 컬럼:", SOURCE_COL)
# print("수정일 컬럼:", UPDATED_AT_COL)


## 4. RAG용 FAQ 정규화

원본 엑셀은 수정하지 않고, RAG에서 쓰기 좋은 형태로 별도 파일을 만듭니다.


In [ ]:
# =========================
# 4. FAQ 정규화
# =========================

df = df_raw.copy()

def safe_get(row, col):
    if col is None:
        return ""
    return str(row.get(col, "")).strip()

normalized_rows = []

for idx, row in df.iterrows():
    question = safe_get(row, QUESTION_COL)
    answer = safe_get(row, ANSWER_COL)

    if not question and not answer:
        continue

    faq_id = f"FAQ-{idx+1:04d}"
    category = safe_get(row, CATEGORY_COL)
    keywords = safe_get(row, KEYWORD_COL)
    source = safe_get(row, SOURCE_COL) or FAQ_PATH.name
    updated_at = safe_get(row, UPDATED_AT_COL)

    search_text = "\n".join([
        f"[카테고리] {category}",
        f"[질문] {question}",
        # f"[키워드] {keywords}",
        f"[답변] {answer}",
    ])

    normalized_rows.append({
        "faq_id": faq_id,
        "category": category,
        "question": question,
        "answer": answer,
        # "keywords": keywords,
        # "source": source,
        # "updated_at": updated_at,
        "search_text": search_text,
        "source_row_number": idx + 2,  # 엑셀 헤더를 고려한 원본 행 번호
    })

faq_df = pd.DataFrame(normalized_rows)

print("정규화 FAQ 건수:", len(faq_df))
display(faq_df.head(5))

faq_df.to_excel(NORMALIZED_FAQ_PATH, index=False)
print("저장 완료:", NORMALIZED_FAQ_PATH)


## 5. 검색 인덱스 만들기

v0에서는 별도 벡터DB 없이 `TF-IDF 문자 n-gram 검색`으로 시작합니다.

한국어는 띄어쓰기와 형태소 분석 이슈가 있어, 초반 검증용으로는 문자 n-gram 방식이 꽤 안정적입니다.


In [ ]:
# =========================
# 5. 검색 인덱스
# =========================

documents = faq_df["search_text"].fillna("").tolist()

# 한국어 FAQ 검색용 초간단 baseline
# analyzer='char_wb': 단어 경계 기반 문자 n-gram
# ngram_range=(2, 5): 2~5글자 단위 검색
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1
)

doc_matrix = vectorizer.fit_transform(documents)

print("문서 수:", doc_matrix.shape[0])
print("특징 수:", doc_matrix.shape[1])


## 6. 검색 함수

질문을 넣으면 관련 FAQ Top-K를 반환합니다.


In [ ]:
# =========================
# 6. 검색 함수
# =========================

def search_faq(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    result = faq_df.iloc[top_indices].copy()
    result.insert(0, "score", scores[top_indices])
    result = result[[
        "score",
        "faq_id",
        "category",
        "question",
        "answer",
        # "keywords",
        # "source",
        # "updated_at",
        "source_row_number",
    ]]
    return result

# 테스트
test_query = "세금계산서는 언제 발행되나요?"
result = search_faq(test_query, top_k=5)

print("질문:", test_query)
display(result)


## 7. 테스트 질문 여러 개 돌려보기

검색 품질을 먼저 확인합니다.  
여기서 검색 Top-5가 엉뚱하면 LLM을 붙여도 답변이 좋아지지 않습니다.


In [ ]:
# =========================
# 7. 검색 테스트
# =========================

test_queries = [
    "세금계산서는 언제 발행되나요?",
    "현금영수증 발급 가능한가요?",
    "배송은 얼마나 걸리나요?",
    "인쇄 시안은 언제 확인할 수 있나요?",
    "로고 파일은 어떤 형식으로 보내야 하나요?",
    "주문 취소는 어떻게 하나요?",
    "견적서는 받을 수 있나요?",
    "대량 주문 할인 가능한가요?",
    "최소 주문 수량이 있나요?",
    "회원가입을 해야 주문할 수 있나요?",
]

test_rows = []

for query in test_queries:
    top = search_faq(query, top_k=5)
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        test_rows.append({
            "query": query,
            "rank": rank,
            "score": row["score"],
            "faq_id": row["faq_id"],
            # "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "source_row_number": row["source_row_number"],
        })

search_test_df = pd.DataFrame(test_rows)
display(search_test_df.head(20))

search_test_df.to_excel(SEARCH_TEST_RESULT_PATH, index=False)
print("검색 테스트 결과 저장:", SEARCH_TEST_RESULT_PATH)


## 8. 근거 기반 답변 초안 생성 함수

아직 LLM을 붙이지 않고, 검색된 FAQ를 근거로 답변 초안을 만드는 방식입니다.

이 단계의 목적은 “검색된 근거가 답변에 쓸 만한지” 확인하는 것입니다.


In [ ]:
# =========================
# 8. 근거 기반 답변 초안
# =========================

def make_grounded_draft_answer(query: str, top_k: int = 3, min_score: float = 0.05) -> str:
    results = search_faq(query, top_k=top_k)

    if results.empty or results["score"].max() < min_score:
        return (
            "관련 FAQ를 충분히 찾지 못했습니다. "
            "정확한 안내를 위해 상담원 확인이 필요합니다."
        )

    lines = []
    lines.append(f"질문: {query}")
    lines.append("")
    lines.append("아래 FAQ를 근거로 답변을 작성할 수 있습니다.")
    lines.append("")

    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        lines.append(f"[근거 {rank}] {row['faq_id']} / 점수 {row['score']:.4f}")
        if row["category"]:
            lines.append(f"- 카테고리: {row['category']}")
        lines.append(f"- FAQ 질문: {row['question']}")
        lines.append(f"- FAQ 답변: {row['answer']}")
        lines.append(f"- 원본 행 번호: {row['source_row_number']}")
        lines.append("")

    return "\n".join(lines)

print(make_grounded_draft_answer("세금계산서는 언제 발행되나요?", top_k=3))


## 9. LLM 답변 생성

검색된 FAQ를 근거로 답변을 생성합니다. 아래 두 가지 백엔드 중 하나를 사용하세요.

- 9-1. Ollama: API 키 없이 로컬 모델 사용 (기본)
- 9-2. OpenRouter: API 키가 있으면 외부 모델 사용 (선택사항)

공통 규칙:

- 검색된 context 안에 있는 내용만 답변
- context에 없으면 모른다고 답변
- FAQ ID를 답변 마지막에 표시
- 정책/결제/세금계산서/환불은 단정하지 않기


In [ ]:
# =========================
# 9-0. LLM 답변 생성 공통 함수
# =========================
# Ollama, OpenRouter 두 백엔드가 공통으로 사용하는 함수입니다.
# 이 셀은 두 백엔드 셀보다 먼저 실행되어야 합니다.

def build_context(results: pd.DataFrame) -> str:
    chunks = []

    for _, row in results.iterrows():
        chunks.append(
            f"""[FAQ ID: {row['faq_id']}]
카테고리: {row['category']}
원본 행 번호: {row['source_row_number']}
질문: {row['question']}
답변: {row['answer']}
"""
        )

    return "\n---\n".join(chunks)


In [ ]:
# =========================
# 9-2. OpenRouter LLM 답변 생성 (선택사항)
# =========================
# API 키가 있으면 OpenRouter로 답변을 생성하고, 없으면 프롬프트만 출력합니다.
# 사용 방법: .env 또는 환경변수에 OPENROUTER_API_KEY를 설정하세요.

import os
import requests

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL = "openai/gpt-4o-mini"  # 사용 가능한 OpenRouter 모델명으로 변경 가능


def generate_answer_with_openrouter(query: str, top_k: int = 3, min_score: float = 0.05, model: str = OPENROUTER_MODEL) -> dict:
    results = search_faq(query, top_k=top_k)

    if results.empty or results["score"].max() < min_score:
        return {
            "query": query,
            "answer": "관련 FAQ를 충분히 찾지 못했습니다. 정확한 안내를 위해 상담원 확인이 필요합니다.",
            "retrieved": results,
        }

    context = build_context(results)

    system_prompt = """
너는 기프트코 고객센터 FAQ 어시스턴트다.

규칙:
- 제공된 context 안에 있는 내용만 근거로 답변한다.
- context에 없는 내용은 추측하지 않는다.
- 답변 마지막에 사용한 FAQ ID를 표시한다.
- 세금계산서, 환불, 결제, 배송 정책은 단정하지 말고 원문 기준으로 답변한다.
- 검색 결과가 부족하면 "상담원 확인이 필요합니다"라고 답변한다.
- 고객에게 보여줄 수 있는 자연스러운 한국어로 답변한다.
"""

    user_prompt = f"""
[사용자 질문]
{query}

[검색된 FAQ Context]
{context}

위 context만 근거로 고객에게 답변해줘.
"""

    if not OPENROUTER_API_KEY:
        print("OPENROUTER_API_KEY가 없어 LLM 호출은 건너뜁니다. 아래는 전달될 프롬프트입니다.")
        print("=" * 80)
        print(user_prompt)
        return {
            "query": query,
            "answer": None,
            "retrieved": results,
            "prompt": user_prompt,
        }

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()},
            ],
            "temperature": 0.2,
        },
        timeout=60,
    )
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]

    return {
        "query": query,
        "answer": answer,
        "retrieved": results,
    }


# 테스트 (API 키 없으면 프롬프트만 출력됨)
result = generate_answer_with_openrouter("세금계산서는 언제 발행되나요?", top_k=3)
if result["answer"]:
    print(result["answer"])
display(result["retrieved"])

## 10. 평가 템플릿 만들기

RAG는 체감이 아니라 평가셋으로 개선해야 합니다.

아래 파일에 `expected_faq_id`를 사람이 채워 넣으면, 검색 품질을 Hit@K로 측정할 수 있습니다.


In [ ]:
# =========================
# 10. 평가 템플릿
# =========================

eval_template = pd.DataFrame({
    "query": test_queries,
    "expected_faq_id": "",
    "expected_keywords": "",
    "memo": "",
})

# eval_template.to_excel(EVAL_TEMPLATE_PATH, index=False)
# print("평가 템플릿 저장:", EVAL_TEMPLATE_PATH)
display(eval_template)


In [ ]:
# =========================
# 평가 템플릿 + 검색 후보 같이 만들기
# =========================

candidate_rows = []

for query in test_queries:
    results = search_faq(query, top_k=5)

    row = {
        "query": query,
        "expected_faq_id": "",
        "expected_keywords": "",
        "memo": "",
    }

    for rank, (_, r) in enumerate(results.iterrows(), start=1):
        row[f"top{rank}_faq_id"] = r["faq_id"]
        row[f"top{rank}_score"] = r["score"]
        row[f"top{rank}_question"] = r["question"]
        row[f"top{rank}_answer"] = r["answer"]

    candidate_rows.append(row)

eval_candidate_df = pd.DataFrame(candidate_rows)

EVAL_CANDIDATE_PATH = OUTPUT_DIR / "giftco_faq_eval_template_with_candidates_v0.xlsx"
eval_candidate_df.to_excel(EVAL_CANDIDATE_PATH, index=False)

print("검수용 평가 템플릿 저장:", EVAL_CANDIDATE_PATH)
display(eval_candidate_df)

In [ ]:
# =========================
# top1 검색 결과를 expected_faq_id로 자동 입력
# =========================

auto_eval_rows = []

for query in test_queries:
    results = search_faq(query, top_k=5)

    if len(results) == 0:
        auto_eval_rows.append({
            "query": query,
            "expected_faq_id": "",
            "expected_keywords": "",
            "memo": "검색 결과 없음",
        })
        continue

    top1 = results.iloc[0]

    auto_eval_rows.append({
        "query": query,
        "expected_faq_id": top1["faq_id"],
        "expected_keywords": "",
        "memo": "임시값: 검색 Top1을 정답으로 자동 입력",
        "top1_score": top1["score"],
        "top1_question": top1["question"],
        "top1_answer": top1["answer"],
    })

eval_df = pd.DataFrame(auto_eval_rows)

AUTO_EVAL_PATH = OUTPUT_DIR / "giftco_faq_eval_template_top1_auto_v0.xlsx"
eval_df.to_excel(AUTO_EVAL_PATH, index=False)

print("Top1 자동 입력 평가 파일 저장:", AUTO_EVAL_PATH)
display(eval_df)

## 11. Hit@K 평가 함수

`expected_faq_id`를 채운 뒤 실행하면 검색 성능을 볼 수 있습니다.


In [ ]:
# =========================
# 11. Hit@K 평가
# =========================

def evaluate_hit_at_k(eval_df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    rows = []

    for _, row in eval_df.iterrows():
        query = str(row.get("query", "")).strip()
        expected = str(row.get("expected_faq_id", "")).strip()

        # 질문이 없으면 제외
        if not query:
            continue

        # expected_faq_id가 비어 있으면 평가 불가로 기록
        if not expected:
            rows.append({
                "query": query,
                "expected_faq_id": "",
                f"retrieved_top_{k}": "",
                f"hit@{k}": None,
                "top1_faq_id": "",
                "top1_score": 0,
                "status": "expected_faq_id 미입력",
            })
            continue

        results = search_faq(query, top_k=k)
        retrieved_ids = results["faq_id"].tolist()

        rows.append({
            "query": query,
            "expected_faq_id": expected,
            f"retrieved_top_{k}": ", ".join(retrieved_ids),
            f"hit@{k}": expected in retrieved_ids,
            "top1_faq_id": retrieved_ids[0] if retrieved_ids else "",
            "top1_score": float(results.iloc[0]["score"]) if len(results) > 0 else 0,
            "status": "평가 완료",
        })

    return pd.DataFrame(rows)

# 사용 예시:
# eval_df = pd.read_excel(EVAL_TEMPLATE_PATH, dtype=str).fillna("")
# eval_result_df = evaluate_hit_at_k(eval_df, k=5)
# display(eval_result_df)
# print("Hit@5:", eval_result_df["hit@5"].mean())

In [ ]:
# eval_df = pd.read_excel(EVAL_TEMPLATE_PATH, dtype=str).fillna("")
eval_result_df = evaluate_hit_at_k(eval_df, k=5)

display(eval_result_df)

hit_col = "hit@5"

valid_eval_df = eval_result_df[eval_result_df[hit_col].notna()]

if len(valid_eval_df) == 0:
    print("아직 expected_faq_id가 입력된 평가 항목이 없습니다.")
    print("EVAL_TEMPLATE_PATH 파일에서 expected_faq_id를 먼저 채워주세요.")
else:
    print("Hit@5:", valid_eval_df[hit_col].mean())

# 다음 단계

v0에서 확인할 것:

1. FAQ 컬럼이 잘 잡혔는가?
2. 검색 결과 Top-5에 정답 FAQ가 들어오는가?
3. 세금계산서/배송/인쇄/견적/취소 등 주요 질문에서 엉뚱한 FAQ가 나오지 않는가?
4. 검색 실패 시 상담원 연결 문구가 나오는가?
5. 평가 템플릿에 정답 FAQ ID를 채울 수 있는가?

v0가 안정되면 다음 단계로 확장합니다.

```text
v0: FAQ 엑셀 + 검색 Top-K 확인
↓
v1: LLM 답변 생성
↓
v2: Chroma 또는 Qdrant 벡터DB 적용
↓
v3: BM25 + Vector Hybrid Search
↓
v4: Reranker 적용
↓
v5: FastAPI 서버화
↓
v6: 상품 추천 RAG와 연결
```
